In [1]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, TypedDict
os.chdir(Path.cwd().parent)
sys.path.append(str(Path.cwd() / "code"))


In [2]:
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langgraph.graph import END, START, StateGraph
from price_scrape_helpers import _brand_market_key, load_market_map

market_file = "code/market_cd.json"
market_map = load_market_map(market_file)
load_dotenv()

SOURCE_OUTPUT_FILE = Path('output\price_check_results_20260426_Fendi.csv')
REFRESHED_OUTPUT_FILE = Path('output\price_check_results_refreshed.csv')
BATCH_SIZE = 12
MODEL_NAME = 'gemini-3.1-pro-preview'

if not SOURCE_OUTPUT_FILE.exists():
    candidates = sorted(
        SOURCE_OUTPUT_FILE.parent.glob('price_check_results_*.csv'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if candidates:
        SOURCE_OUTPUT_FILE = candidates[0]
        print(f'Warning: source file not found, using latest available file: {SOURCE_OUTPUT_FILE}')
    else:
        raise FileNotFoundError(f'Missing source file: {SOURCE_OUTPUT_FILE}')

df_results = pd.read_csv(SOURCE_OUTPUT_FILE)
df_results.columns = [str(c).strip() for c in df_results.columns]

failed_mask = df_results.get('new_price', pd.Series(index=df_results.index, dtype='float64')).isna()
df_failed = df_results.loc[failed_mask].copy()

print('Source rows:', len(df_results))
print('Failed rows needing refresh:', len(df_failed))

client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
search_tool = types.Tool(google_search=types.GoogleSearch())

def _chunk_rows(frame, size):
    for start in range(0, len(frame), size):
        yield frame.iloc[start:start + size]

Source rows: 80
Failed rows needing refresh: 50


In [5]:
df = pd.read_csv("data/price_results_20260606_145609.csv")
df.shape

(704, 17)

In [ ]:
from handlers.updator import run_url_update_pipeline as _run_url_update_pipeline
_run_url_update_pipeline(
    input_path=str(SOURCE_OUTPUT_FILE),
    output_path=str(REFRESHED_OUTPUT_FILE),
    target_scope={},  # No filtering, process all rows
    batch_size=BATCH_SIZE,
    progress_callback=None,  # You can implement a callback to track progress if needed
    only_without_price=True,  # Only refresh rows where 'new_price' is NaN
)

In [ ]:
def _extract_json_block(text):
    cleaned = str(text).strip()
    if cleaned.startswith('```json') and cleaned.endswith('```'):
        cleaned = cleaned.replace('```json', '', 1).rsplit('```', 1)[0].strip()
    elif cleaned.startswith('```') and cleaned.endswith('```'):
        cleaned = cleaned.replace('```', '', 1).rsplit('```', 1)[0].strip()

    first_arr = cleaned.find('[')
    last_arr = cleaned.rfind(']')
    if first_arr != -1 and last_arr != -1 and last_arr > first_arr:
        return cleaned[first_arr:last_arr + 1]

    first_obj = cleaned.find('{')
    last_obj = cleaned.rfind('}')
    if first_obj != -1 and last_obj != -1 and last_obj > first_obj:
        return cleaned[first_obj:last_obj + 1]

    return cleaned

class UrlGraphState(TypedDict, total=False):
    row: dict[str, Any]
    refresh_item: dict[str, Any]
    validated_item: dict[str, Any]
    error: str

def _refresh_url(state: UrlGraphState) -> UrlGraphState:
    row = state.get('row', {})
    brand = row.get('brand')
    key = _brand_market_key(brand, market_map)
    locale_market_map = market_map.get(key, {})

    prompt = f"""
        # task: find the web page url for the input content
        Input fields are: brand, name, skus, market.
        If unresolved, set new_url to null and explain the reason.
        do the following steps:
        1. find the default url given brand, name, and skus.
        2. modify default url to certain market following the market_code_map of this brand.
        3. save the modified url in new_url.
        4. mark url_status as "resolved" if the url is correct and working, otherwise "failed".

        # market_code_map: {locale_market_map}
        # input: {row}

        # output ONLY valid JSON format, example:
        {{'brand': 'Van Cleef',
        'new_name': 'Vintage Alhambra bracelet, 5 motifs',
        'sku': 'VCARA41300',
        'market': 'VNM',
        'new_url': 'https://www.vancleefarpels.com/vn/en/collections/jewelry/bracelets/vintage-alhambra/VARA41300.html',
        'url_status': 'resolved',
        'reason':  (null if resolved else a string explaining why)}}
        """
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
        tools=[search_tool],
        thinking_config=types.ThinkingConfig(
            thinking_level="LOW"
        )
    ),
    )
    text = response.text or ''
    payload = json.loads(_extract_json_block(text))
    if isinstance(payload, list):
        payload = payload[0] if payload else {}
    if not isinstance(payload, dict):
        raise ValueError('Refresh response must be a JSON object or list with one object.')
    return {'refresh_item': payload}

def _validation_url(state: UrlGraphState) -> UrlGraphState:
    row = state.get('row', {})
    refresh_item = state.get('refresh_item', {})
    prompt_input = {
        'brand': row.get('brand'),
        'market': row.get('market'),
        'skus': row.get('skus'),
        'name': row.get('name'),
        'new_url': refresh_item.get('new_url'),
        'url_status': refresh_item.get('url_status', refresh_item.get('status')),
        'reason': refresh_item.get('reason'),
    }

    prompt = (
        'open the url and check if it is the corresponding product page (right sku, market) '
        'if not, update the url to a right one. Return ONLY valid JSON with keys: '
        'brand, market, sku, new_url, url_status(resolved/failed), reason. '
        '# input: '
        f"{json.dumps(prompt_input, ensure_ascii=False)}"
    )

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
        tools=[search_tool],
        thinking_config=types.ThinkingConfig(
            thinking_level="LOW"
        )
    ),
    )
    text = response.text or ''
    payload = json.loads(_extract_json_block(text))
    if isinstance(payload, list):
        payload = payload[0] if payload else {}
    if not isinstance(payload, dict):
        raise ValueError('Validation response must be a JSON object or list with one object.')
    return {'validated_item': payload}

url_graph_builder = StateGraph(UrlGraphState)
url_graph_builder.add_node('refresh', _refresh_url)
url_graph_builder.add_node('validation', _validation_url)
url_graph_builder.add_edge(START, 'refresh')
url_graph_builder.add_edge('refresh', 'validation')
url_graph_builder.add_edge('validation', END)
url_graph = url_graph_builder.compile()

def _run_url_graph(batch_frame):
    state = {'row': batch_frame.to_dict()}
    return url_graph.invoke(state)

In [29]:
refresh_rows = []

for batch_index, raw_batch in enumerate(_chunk_rows(df_failed, BATCH_SIZE), start=1):
    if raw_batch.empty:
        continue

    for row_idx, row in raw_batch.iterrows():
        out = {
            '__src_idx': row_idx,
            'updated_url': pd.NA,
            'url_status': pd.NA,
        }

        row_frame = pd.Series(
            {
                'brand': row.get('brand'),
                'name': row.get('name'),
                'sku': row.get('skus'),
                'market': row.get('market'),
            }
        )
        graph_state = _run_url_graph(row_frame)
        refresh_item = graph_state.get('refresh_item', {})
        validated_item = graph_state.get('validated_item', {})

        out['updated_url'] = validated_item.get('new_url', refresh_item.get('new_url', pd.NA))
        out['url_status'] = validated_item.get('url_status', refresh_item.get('url_status', refresh_item.get('status', pd.NA)))

        refresh_rows.append(out)

payload_df = pd.DataFrame(refresh_rows)

df_failed_refreshed = df_failed.copy()
if not payload_df.empty:
    payload_df = payload_df.set_index('__src_idx')
    common_idx = df_failed_refreshed.index.intersection(payload_df.index)

    for col in ['updated_url', 'url_status']:
        if col not in df_failed_refreshed.columns:
            df_failed_refreshed[col] = pd.NA
        df_failed_refreshed.loc[common_idx, col] = payload_df.loc[common_idx, col]

print('Refreshed rows:', len(df_failed_refreshed))
print('Rows with updated_url:', int(df_failed_refreshed['updated_url'].notna().sum()))

Raw response from Gemini for refresh: ```json
{
  "brand": "Dior",
  "new_name": "Miss Caro Top Handle Pouch",
  "sku": "S5221UHAG_M900",
  "market": "NZL",
  "new_url": "https://www.dior.com/en_nz/fashion/products/S5221UHAG_M900",
  "url_status": "resolved",
  "reason": null
}
```
Prompt tokens: 586
Output tokens: 114
Total: 1554
Raw response from Gemini for validation: ```json
{
  "brand": "Dior",
  "market": "NZL",
  "sku": "S5221UHAG_M900",
  "new_url": "https://www.dior.com/en_nz/fashion/products/S5221UHAG_M900",
  "url_status": "resolved",
  "reason": "The provided URL is correct and valid for the Miss Caro Top Handle Pouch with SKU S5221UHAG_M900 for the New Zealand market."
}
```
Prompt tokens: 140
Output tokens: 134
Total: 2330
Raw response from Gemini for refresh: ```json
{
  "brand": "Dior",
  "new_name": "Dior Saddle Bag Mini",
  "sku": "M0456CBAA_M900",
  "market": "BRA",
  "new_url": "https://www.dior.com/pt_br/fashion/products/M0456CBAA_M900",
  "url_status": "resolved",

C:\Users\p2cao\AppData\Local\Temp\ipykernel_57056\1088112159.py:50: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[<NA> <NA> <NA> <NA> <NA> <NA> <NA> <NA>]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_failed_refreshed.loc[common_idx, col] = payload_df.loc[common_idx, col]
